# ĐỒ ÁN MÔN HỌC: XỬ LÝ ẢNH — NHẬN DIỆN CHẤT LƯỢNG GẠO
## Phân hệ 3: Trích xuất đặc trưng hạt gạo (Feature Extraction) — Tuân Đoàn

Notebook này thực hiện trích xuất đặc trưng định lượng cho ảnh gạo mẫu đơn lẻ và từng hạt gạo của người dùng, kết hợp đặc trưng **Hình dạng**, **Màu sắc** và **Kết cấu** bằng phương pháp xử lý ảnh thuần túy[cite: 4, 5, 6].

**Đặc tả kỹ thuật:**
* **Đặc trưng về hình dạng (`Shape`):** Tính diện tích (`cv2.contourArea`), chu vi (`cv2.arcLength`), chiều dài/rộng và tỷ lệ khung bao xoay tối thiểu (`cv2.minAreaRect`), độ tròn (`Circularity`) và độ đặc (`Solidity`)[cite: 4].
* **Đặc trưng về màu sắc (`Color`):** Tính giá trị `RGB` trung bình bên trong vùng hạt trên ảnh màu đã tiền xử lý, chuyển sang hệ màu `HSV` để đo độ sáng (`V`) và độ bão hòa (`S`), tính tỷ lệ pixel vàng hoặc sẫm màu và xây dựng vector đặc trưng màu[cite: 4, 5].
* **Đặc trưng về kết cấu (`Texture - GLCM`):** Tính ma trận đồng xuất hiện mức xám (`GLCM`) với các chỉ số `Contrast`, `Homogeneity`, `Energy` và `Correlation`[cite: 4].
* **Tổng hợp & kiểm tra dữ liệu:** Ghép `Shape + Color + Texture` thành vector đặc trưng cho từng hạt, kiểm tra dữ liệu thiếu/bất thường và xuất các tệp `.csv` đặc trưng[cite: 4].

In [ ]:
# %% [code]
import os
import cv2
import numpy as np
import pandas as pd
from skimage.feature import graycomatrix, graycoprops

try:
    PROJECT_ROOT = os.path.dirname(os.path.abspath(__file__))
except NameError:
    PROJECT_ROOT = os.getcwd()

# Dữ liệu từ TV1 (ảnh màu đã tiền xử lý cho cả mẫu đơn lẻ và mẻ gạo người dùng)
REF_COLOR_DIR = os.path.join(
    PROJECT_ROOT, "processed_dataset", "1_reference_labeled", "color_clean"
)
REF_CSV_PATH = os.path.join(
    PROJECT_ROOT, "processed_dataset", "1_reference_labeled", "reference_labels.csv"
)
BATCH_COLOR_DIR = os.path.join(
    PROJECT_ROOT, "processed_dataset", "2_batch_rice_images", "color_clean"
)

# Dữ liệu từ TV2 (bảng phân đoạn & vùng hạt cho ảnh mẻ gạo người dùng)
SEG_INDEX_CSV = os.path.join(
    PROJECT_ROOT, "segmented_dataset", "segmented_grains.csv"
)

# Đầu ra của TV3 (tách rõ đặc trưng mẫu đơn lẻ và đặc trưng hạt người dùng)
FEAT_OUTPUT_DIR = os.path.join(PROJECT_ROOT, "features_dataset")
REF_FEAT_CSV_PATH = os.path.join(FEAT_OUTPUT_DIR, "reference_grain_features.csv")
BATCH_FEAT_CSV_PATH = os.path.join(FEAT_OUTPUT_DIR, "grain_features.csv")
BATCH_FEAT_CLEAN_CSV_PATH = os.path.join(FEAT_OUTPUT_DIR, "grain_features_clean.csv")

REF_FOLDER_TO_LABEL = {
    "Tot": "Tốt",
    "Vo": "Vỡ",
    "Kem": "Kém",
    "Te": "Tệ",
    "Tap_chat": "Tạp chất",
}
SUPPORTED_EXTS = (".jpg", ".jpeg", ".png", ".bmp")

FEAT_CONFIG = {
    "yellow_h_min": 15,
    "yellow_h_max": 38,
    "yellow_s_min": 40,
    "yellow_v_min": 60,
    "dark_v_max": 85,
    "dark_s_min": 30,
    "glcm_levels": 64,
    "glcm_distances": [1, 2],
    "glcm_angles_deg": [0, 45, 90, 135],
}

print(f"Thư mục gốc: {PROJECT_ROOT}")

In [5]:
# %% [code]
def compute_shape_metrics(mask: np.ndarray, override_area: float | None = None) -> dict:
    """
    Tính đặc trưng hình dạng chuẩn từ mask vùng hạt:
    area, perimeter, length, width, aspect_ratio, circularity, solidity.
    """
    if mask is None or mask.size == 0:
        return {
            "shape_area": np.nan,
            "shape_perimeter": np.nan,
            "shape_length": np.nan,
            "shape_width": np.nan,
            "shape_aspect_ratio": np.nan,
            "shape_circularity": np.nan,
            "shape_solidity": np.nan,
        }

    binary = (mask > 0).astype(np.uint8)
    contours, _ = cv2.findContours(binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours:
        return {
            "shape_area": np.nan,
            "shape_perimeter": np.nan,
            "shape_length": np.nan,
            "shape_width": np.nan,
            "shape_aspect_ratio": np.nan,
            "shape_circularity": np.nan,
            "shape_solidity": np.nan,
        }

    cnt = max(contours, key=cv2.contourArea)
    cnt_area = float(cv2.contourArea(cnt))
    area = float(override_area) if (override_area is not None and not pd.isna(override_area) and float(override_area) > 0) else cnt_area
    perimeter = float(cv2.arcLength(cnt, True))

    (_, _), (rw, rh), _ = cv2.minAreaRect(cnt)
    length = float(max(rw, rh))
    width = float(min(rw, rh))
    aspect_ratio = float(length / width) if width > 1e-6 else np.nan

    circularity = float((4.0 * np.pi * cnt_area) / (perimeter * perimeter)) if perimeter > 1e-6 else np.nan

    hull = cv2.convexHull(cnt)
    hull_area = float(cv2.contourArea(hull))
    solidity = float(cnt_area / hull_area) if hull_area > 1e-6 else np.nan

    return {
        "shape_area": round(area, 4),
        "shape_perimeter": round(perimeter, 4),
        "shape_length": round(length, 4),
        "shape_width": round(width, 4),
        "shape_aspect_ratio": round(aspect_ratio, 4) if not np.isnan(aspect_ratio) else np.nan,
        "shape_circularity": round(circularity, 4) if not np.isnan(circularity) else np.nan,
        "shape_solidity": round(solidity, 4) if not np.isnan(solidity) else np.nan,
    }


def compute_color_metrics(bgr_patch: np.ndarray, mask: np.ndarray, cfg: dict = FEAT_CONFIG) -> tuple[dict, np.ndarray]:
    """
    Tính đặc trưng màu sắc (RGB, HSV, V, S, tỷ lệ vàng/sẫm, vector màu) trên vùng hạt.
    """
    if bgr_patch is None or mask is None or bgr_patch.size == 0 or mask.size == 0:
        empty = {
            "color_r_mean": np.nan,
            "color_g_mean": np.nan,
            "color_b_mean": np.nan,
            "color_h_mean": np.nan,
            "color_s_mean": np.nan,
            "color_v_mean": np.nan,
            "color_s_std": np.nan,
            "color_v_std": np.nan,
            "color_yellow_ratio": np.nan,
            "color_dark_ratio": np.nan,
        }
        return empty, np.full(8, np.nan, dtype=np.float64)

    if bgr_patch.shape[:2] != mask.shape[:2]:
        mask = cv2.resize(mask, (bgr_patch.shape[1], bgr_patch.shape[0]), interpolation=cv2.INTER_NEAREST)

    mask_bool = mask > 0
    num_px = int(np.count_nonzero(mask_bool))
    if num_px == 0:
        mask_bool = np.ones(bgr_patch.shape[:2], dtype=bool)
        num_px = int(np.count_nonzero(mask_bool))

    rgb = cv2.cvtColor(bgr_patch, cv2.COLOR_BGR2RGB)
    hsv = cv2.cvtColor(bgr_patch, cv2.COLOR_BGR2HSV)

    r_vals = rgb[:, :, 0][mask_bool].astype(np.float64)
    g_vals = rgb[:, :, 1][mask_bool].astype(np.float64)
    b_vals = rgb[:, :, 2][mask_bool].astype(np.float64)

    h_vals = hsv[:, :, 0][mask_bool].astype(np.float64)
    s_vals = hsv[:, :, 1][mask_bool].astype(np.float64)
    v_vals = hsv[:, :, 2][mask_bool].astype(np.float64)

    r_mean, g_mean, b_mean = float(np.mean(r_vals)), float(np.mean(g_vals)), float(np.mean(b_vals))
    h_mean, s_mean, v_mean = float(np.mean(h_vals)), float(np.mean(s_vals)), float(np.mean(v_vals))
    s_std, v_std = float(np.std(s_vals)), float(np.std(v_vals))

    yellow_px = (
        (h_vals >= cfg["yellow_h_min"])
        & (h_vals <= cfg["yellow_h_max"])
        & (s_vals >= cfg["yellow_s_min"])
        & (v_vals >= cfg["yellow_v_min"])
    )
    dark_px = (v_vals <= cfg["dark_v_max"]) | ((v_vals <= 110) & (s_vals >= cfg["dark_s_min"]))

    yellow_ratio = float(np.count_nonzero(yellow_px)) / float(num_px)
    dark_ratio = float(np.count_nonzero(dark_px)) / float(num_px)

    feats = {
        "color_r_mean": round(r_mean, 4),
        "color_g_mean": round(g_mean, 4),
        "color_b_mean": round(b_mean, 4),
        "color_h_mean": round(h_mean, 4),
        "color_s_mean": round(s_mean, 4),
        "color_v_mean": round(v_mean, 4),
        "color_s_std": round(s_std, 4),
        "color_v_std": round(v_std, 4),
        "color_yellow_ratio": round(yellow_ratio, 6),
        "color_dark_ratio": round(dark_ratio, 6),
    }

    vec = np.array([
        r_mean / 255.0,
        g_mean / 255.0,
        b_mean / 255.0,
        h_mean / 179.0,
        s_mean / 255.0,
        v_mean / 255.0,
        yellow_ratio,
        dark_ratio,
    ], dtype=np.float64)

    return feats, vec


def compute_glcm_metrics(gray_patch: np.ndarray, mask: np.ndarray, cfg: dict = FEAT_CONFIG) -> dict:
    """
    Tính đặc trưng kết cấu GLCM (Contrast, Homogeneity, Energy, Correlation) trên vùng hạt.
    """
    if gray_patch is None or mask is None or gray_patch.size == 0 or mask.size == 0:
        return {
            "glcm_contrast": np.nan,
            "glcm_homogeneity": np.nan,
            "glcm_energy": np.nan,
            "glcm_correlation": np.nan,
        }

    if gray_patch.shape[:2] != mask.shape[:2]:
        mask = cv2.resize(mask, (gray_patch.shape[1], gray_patch.shape[0]), interpolation=cv2.INTER_NEAREST)

    mask_bool = mask > 0
    if np.count_nonzero(mask_bool) < 4:
        return {
            "glcm_contrast": np.nan,
            "glcm_homogeneity": np.nan,
            "glcm_energy": np.nan,
            "glcm_correlation": np.nan,
        }

    gray_2d = gray_patch if gray_patch.ndim == 2 else cv2.cvtColor(gray_patch, cv2.COLOR_BGR2GRAY)
    mean_val = float(np.mean(gray_2d[mask_bool]))
    clean_gray = np.clip(np.where(mask_bool, gray_2d.astype(np.float64), mean_val), 0, 255).astype(np.uint8)

    levels = int(cfg.get("glcm_levels", 64))
    quantized = np.clip(np.floor(clean_gray.astype(np.float32) * (levels / 256.0)).astype(np.uint8), 0, levels - 1)
    angles_rad = [np.deg2rad(d) for d in cfg.get("glcm_angles_deg", [0, 45, 90, 135])]
    distances = cfg.get("glcm_distances", [1, 2])

    glcm = graycomatrix(quantized, distances=distances, angles=angles_rad, levels=levels, symmetric=True, normed=True)
    return {
        "glcm_contrast": round(float(np.mean(graycoprops(glcm, "contrast"))), 6),
        "glcm_homogeneity": round(float(np.mean(graycoprops(glcm, "homogeneity"))), 6),
        "glcm_energy": round(float(np.mean(graycoprops(glcm, "energy"))), 6),
        "glcm_correlation": round(float(np.mean(graycoprops(glcm, "correlation"))), 6),
    }


def assemble_feature_record(base_meta: dict, shape_f: dict, color_f: dict, color_vec: np.ndarray, glcm_f: dict) -> dict:
    combined = np.concatenate([
        np.array([
            shape_f["shape_area"],
            shape_f["shape_perimeter"],
            shape_f["shape_length"],
            shape_f["shape_width"],
            shape_f["shape_aspect_ratio"],
            shape_f["shape_circularity"],
            shape_f["shape_solidity"],
        ], dtype=np.float64),
        color_vec,
        np.array([
            glcm_f["glcm_contrast"],
            glcm_f["glcm_homogeneity"],
            glcm_f["glcm_energy"],
            glcm_f["glcm_correlation"],
        ], dtype=np.float64),
    ])
    abnormal = bool(np.isnan(combined).any() or np.isinf(combined).any())

    rec = dict(base_meta)
    rec.update(shape_f)
    rec.update(color_f)
    rec.update(glcm_f)
    rec["color_vector"] = np.round(color_vec, 6).tolist()
    rec["feature_vector"] = np.round(combined, 6).tolist()
    rec["feature_dim"] = int(len(combined))
    rec["feature_status"] = "abnormal_or_missing" if abnormal else "ok"
    return rec

In [ ]:
# %% [code]
def extract_single_grain_mask_from_color(bgr_img: np.ndarray) -> np.ndarray:
    """
    Tách mask hạt đơn lẻ trên ảnh mẫu đã tiền xử lý màu bằng xử lý ảnh thuần túy.
    """
    gray = cv2.cvtColor(bgr_img, cv2.COLOR_BGR2GRAY)
    blur = cv2.GaussianBlur(gray, (5, 5), 0)
    clahe = cv2.createCLAHE(clipLimit=2.5, tileGridSize=(8, 8))
    enh = clahe.apply(blur)

    _, mask = cv2.threshold(enh, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    if np.count_nonzero(mask) > 0.65 * mask.size:
        mask = cv2.bitwise_not(mask)

    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3))
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel, iterations=1)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel, iterations=1)

    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours:
        return np.ones_like(gray, dtype=np.uint8) * 255

    main_cnt = max(contours, key=cv2.contourArea)
    out_mask = np.zeros_like(gray, dtype=np.uint8)
    cv2.drawContours(out_mask, [main_cnt], -1, 255, thickness=-1)
    return out_mask


def collect_single_grain_reference_items() -> list[dict]:
    items = []
    if os.path.exists(REF_CSV_PATH):
        df_ref = pd.read_csv(REF_CSV_PATH)
        for idx, row in df_ref.iterrows():
            c_path = str(row.get("color_path", "")).strip()
            if c_path and not os.path.isabs(c_path):
                c_path = os.path.join(PROJECT_ROOT, c_path)
            if not c_path or not os.path.exists(c_path):
                continue
            lbl = str(row.get("label_name", "")).strip() or REF_FOLDER_TO_LABEL.get(
                str(row.get("folder_code", "")).strip(), ""
            )
            items.append({
                "sample_id": str(row.get("sample_id", f"ref_{idx+1:04d}")),
                "label_name": lbl,
                "color_path": c_path,
            })

    if not items and os.path.exists(REF_COLOR_DIR):
        for fcode, lbl in REF_FOLDER_TO_LABEL.items():
            sdir = os.path.join(REF_COLOR_DIR, fcode)
            if not os.path.exists(sdir):
                continue
            for idx, fn in enumerate(sorted(os.listdir(sdir)), start=1):
                if fn.lower().endswith(SUPPORTED_EXTS):
                    items.append({
                        "sample_id": f"{fcode}_{idx:04d}",
                        "label_name": lbl,
                        "color_path": os.path.join(sdir, fn),
                    })
    return items


def extract_reference_single_grain_features(cfg: dict = FEAT_CONFIG) -> pd.DataFrame:
    """
    Trích xuất đặc trưng cho từng ảnh gạo đơn lẻ trong dataset mẫu.
    """
    items = collect_single_grain_reference_items()
    rows = []

    for item in items:
        bgr = cv2.imread(item["color_path"])
        if bgr is None:
            continue
        mask = extract_single_grain_mask_from_color(bgr)
        gray = cv2.cvtColor(bgr, cv2.COLOR_BGR2GRAY)

        shape_f = compute_shape_metrics(mask)
        color_f, color_vec = compute_color_metrics(bgr, mask, cfg=cfg)
        glcm_f = compute_glcm_metrics(gray, mask, cfg=cfg)

        meta = {
            "sample_id": item["sample_id"],
            "label_name": item["label_name"],
        }
        rows.append(assemble_feature_record(meta, shape_f, color_f, color_vec, glcm_f))

    df_ref_feat = pd.DataFrame(rows)
    os.makedirs(FEAT_OUTPUT_DIR, exist_ok=True)
    df_ref_feat.to_csv(REF_FEAT_CSV_PATH, index=False, encoding="utf-8-sig")
    print(f"Đã trích xuất {len(df_ref_feat)} ảnh gạo đơn lẻ mẫu: {REF_FEAT_CSV_PATH}")
    return df_ref_feat


def _load_optional_gray(rel_path) -> np.ndarray | None:
    if not rel_path or pd.isna(rel_path):
        return None
    p = str(rel_path).strip()
    fp = p if os.path.isabs(p) else os.path.join(PROJECT_ROOT, p)
    return cv2.imread(fp, cv2.IMREAD_GRAYSCALE) if os.path.exists(fp) else None


def _load_tv1_batch_color(batch_id: str, cache: dict) -> np.ndarray | None:
    if batch_id in cache:
        return cache[batch_id]
    if not os.path.exists(BATCH_COLOR_DIR):
        cache[batch_id] = None
        return None
    cand = os.path.join(BATCH_COLOR_DIR, f"{batch_id}.jpg")
    img_p = cand if os.path.exists(cand) else None
    if img_p is None:
        for fn in sorted(os.listdir(BATCH_COLOR_DIR)):
            if os.path.splitext(fn)[0] == batch_id:
                img_p = os.path.join(BATCH_COLOR_DIR, fn)
                break
    img = cv2.imread(img_p) if img_p and os.path.exists(img_p) else None
    cache[batch_id] = img
    return img


def extract_user_batch_features_from_tv2_and_tv1(cfg: dict = FEAT_CONFIG) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    Trích xuất đặc trưng cho hạt người dùng:
    - Hình dạng & kết cấu từ vùng hạt/CSV của TV2
    - Màu sắc từ ảnh màu đã xử lý của TV1
    """
    if not os.path.exists(SEG_INDEX_CSV):
        raise FileNotFoundError(f"Không tìm thấy {SEG_INDEX_CSV}")

    df_tv2 = pd.read_csv(SEG_INDEX_CSV)
    color_cache = {}
    rows = []

    for _, row in df_tv2.iterrows():
        roi_mask = _load_optional_gray(row.get("roi_mask_path", ""))
        roi_gray = _load_optional_gray(row.get("roi_gray_path", ""))

        if roi_mask is not None:
            th, tw = roi_mask.shape[:2]
        else:
            th = max(1, int(row.get("bbox_h", 16)))
            tw = max(1, int(row.get("bbox_w", 16)))
            roi_mask = np.ones((th, tw), dtype=np.uint8) * 255

        img_color = _load_tv1_batch_color(str(row["batch_id"]), color_cache)
        color_patch = None
        if img_color is not None and img_color.size > 0:
            h_img, w_img = img_color.shape[:2]
            x, y = int(row["bbox_x"]), int(row["bbox_y"])
            w, h = int(row["bbox_w"]), int(row["bbox_h"])
            x1 = max(0, x - max(0, (tw - w) // 2))
            y1 = max(0, y - max(0, (th - h) // 2))
            x2 = min(w_img, x1 + tw)
            y2 = min(h_img, y1 + th)
            patch = img_color[y1:y2, x1:x2]
            if patch.size > 0:
                color_patch = cv2.resize(patch, (tw, th), interpolation=cv2.INTER_NEAREST) if patch.shape[:2] != (th, tw) else patch

        if roi_gray is None and color_patch is not None:
            roi_gray = cv2.cvtColor(color_patch, cv2.COLOR_BGR2GRAY)

        shape_f = compute_shape_metrics(roi_mask, override_area=row.get("area_px", np.nan))
        color_f, color_vec = compute_color_metrics(color_patch, roi_mask, cfg=cfg)
        glcm_f = compute_glcm_metrics(roi_gray, roi_mask, cfg=cfg)

        meta = {
            "grain_uid": row.get("grain_uid", ""),
            "batch_id": row.get("batch_id", ""),
            "grain_id": int(row["grain_id"]) if not pd.isna(row.get("grain_id", np.nan)) else -1,
            "centroid_x": row.get("centroid_x", np.nan),
            "centroid_y": row.get("centroid_y", np.nan),
        }
        rows.append(assemble_feature_record(meta, shape_f, color_f, color_vec, glcm_f))

    df_all = pd.DataFrame(rows)
    df_clean = df_all[df_all["feature_status"] == "ok"].reset_index(drop=True)

    os.makedirs(FEAT_OUTPUT_DIR, exist_ok=True)
    df_all.to_csv(BATCH_FEAT_CSV_PATH, index=False, encoding="utf-8-sig")
    df_clean.to_csv(BATCH_FEAT_CLEAN_CSV_PATH, index=False, encoding="utf-8-sig")

    print(f"Đã trích xuất {len(df_all)} hạt người dùng (hợp lệ: {len(df_clean)}):    {BATCH_FEAT_CLEAN_CSV_PATH}")
    return df_all, df_clean


df_ref_features = extract_reference_single_grain_features()
df_batch_all, df_batch_clean = extract_user_batch_features_from_tv2_and_tv1()